# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR<sup>2</sup> dataset using the `mlcroissant` library, referencing schema entities by their `@id` throughout.

### Dataset Source
The dataset is described by a Croissant JSON-LD schema, available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` and pandas are installed
!pip install -q mlcroissant pandas

## 1. Data Loading

Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Explore available record sets, their fields and columns, referencing them by their `@id`.

In [ ]:
# List all record sets by @id and their fields
print("Available record sets and fields (@id):\n")
for rs in dataset.record_sets:
    print(f"Record Set: {rs['@id']} (name: {rs.get('name', '[unnamed]')})")
    if 'field' in rs:
        field_list = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields:")
        for f in field_list:
            if isinstance(f, dict):
                print(f"    - {f.get('@id', '[no id]')} (name: {f.get('name', '[unnamed]')})")
            else:
                print(f"    - {f}")
    else:
        print("  [No fields listed]")
    print()

## 3. Data Extraction

Load all data for each record set into DataFrames, always referencing by each record set's `@id`. 

> **Note:** Below, we identify record set and field `@id`s by examining the overview output.

In [ ]:
from collections import OrderedDict
import warnings
warnings.filterwarnings('ignore')

# Collect all record set @ids
record_sets = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Load each record set into DataFrame using its @id
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records from: {record_set_id}")
    else:
        print(f"No records found for: {record_set_id}")

# If at least one DataFrame loaded, display its columns
if dataframes:
    first_rs = next(iter(dataframes))
    print(f"\nAvailable columns in {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field (referenced by its `@id`) from the main record set for filtering and normalization. We'll also group by an appropriate group field (`@id`).

In [ ]:
# Identify candidate fields (@id) for numeric and categorical analysis
# For this example, use knowledge of typical clinical variables. Update these as appropriate from the previous overview.

# Assume one main record set -- find a DataFrame with data
main_record_set_id = None
for rs_id, df in dataframes.items():
    # Pick first DataFrame with non-empty numeric columns
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        main_record_set_id = rs_id
        break

if main_record_set_id is None:
    print("No numeric fields available in loaded record sets.")
else:
    df = dataframes[main_record_set_id]
    print(f"Working with record set: {main_record_set_id}\nColumns: {df.columns.tolist()}")
    # If there is an 'Age' field, use that, otherwise use the first numeric column
    if 'Age' in df.columns:
        numeric_field_id = 'Age'
    else:
        numeric_field_id = numeric_cols[0] if len(numeric_cols) > 0 else df.columns[0]
    
    # Example: filter where Age > 60, normalize Age, and group by 'Sex' if present
    threshold = 60
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field (@id)
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} (z-score) for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by 'Sex' (or first available category column)
        group_field_candidates = ['Sex', 'Gender', 'Anatomical location', 'MSI status']
        group_field_id = None
        for gf in group_field_candidates:
            if gf in df.columns:
                group_field_id = gf
                break
        if group_field_id:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped)
        else:
            print("\nNo categorical field found for grouping.")
    else:
        print(f"Field {numeric_field_id} not in dataframe.")

## 5. Visualization

Visualize the distribution of the chosen numeric field and the relationship with the grouping field.

> All fields are referenced by their `@id` (column names in DataFrame, as above).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if EDA created filtered_df
try:
    # Histogram of the numeric variable
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f'Histogram of {numeric_field_id} (@id)')
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group (if any)
    if group_field_id:
        plt.figure(figsize=(7, 4))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} grouped by {group_field_id} (@id)')
        plt.show()
except Exception as e:
    print(f"Visualization skipped due to: {e}")

## 6. Conclusion

In this notebook, we've:
- Loaded and explored the FAIR<sup>2</sup> colorectal cancer survivor dataset by its Croissant schema.
- Referenced all record sets and fields strictly by their `@id` for reproducibility.
- Extracted all record sets, and performed basic EDA on a selected numeric field (e.g., Age), normalizing and grouping the data for summary.
- Visualized the distributions and explained potential clinical utility.

To further analyze or model this data, continue referencing all entities by their Croissant `@id` as shown.